## Read in Libraries

In [26]:
from datetime import datetime
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore') 

## Read in Facebook/Instagram Spend and Impressions Data

In [27]:
# STUDENT INPUT REQUIRED - Modify path directly below for location of the 04a. Facebook - Missing values - Raw data.csv file on your laptop/desktop
raw_data_path =  '04a. Facebook - Missing values - Raw data.csv'
export_data_path = 'Processed Data'
file_name = 'final_facebook.csv'

## Defining import / export file paths

In [28]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation = pd.read_csv(raw_data_path)

data_for_imputation.dtypes

week_starting_date     object
fbig_spend            float64
fbig_imp              float64
dtype: object

## Student Comments
### The data types of each column are shown by this code once it has read the raw Facebook data file.
### Prior to embarkinhg any data cleaning or analysis, it is helpful to determine the data's format (such as date or numeric).
### We can clearly the data set shows week_starting_date     object
### fbig_spend            float64
### fbig_imp              float64
### dtype: object 








In [29]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation['week_starting_date'] = pd.to_datetime(data_for_imputation['week_starting_date'])

# STUDENT INPUT REQUIRED - Use the head function to write code that prints out the first 10 rows of the data_for_imputation dataframe - HINT: see how head funtion used in separate Outdoor Campaigns python script
data_for_imputation.head()

,week_starting_date,fbig_spend,fbig_imp
0,2019-01-07,0.0,0.0
1,2019-01-14,0.0,0.0
2,2019-01-21,0.0,0.0
3,2019-01-28,0.0,0.0
4,2019-02-04,0.0,0.0


### The 'week_starting_date' column is converted from a string format to a correct datetime format by this code, which would support subsequent time-based operations and analysis. 

In [30]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation.loc[(data_for_imputation['fbig_spend']>0)&
                                               (data_for_imputation['fbig_imp']==0)]

,week_starting_date,fbig_spend,fbig_imp
21,2019-06-03,13444.237150,0.0
22,2019-06-10,17661.095060,0.0
23,2019-06-17,5969.195235,0.0
24,2019-06-24,6456.339760,0.0


### In order to identify rows with a Big Facebook expenditure of more than 0 but a corresponding number of impressions of 0, this method filters the dataset on the basis of this criteria.  This perhaps would help aid in finding possible problems with data quality or authenticity, like inconsistent or missing impression data in spite of advertising expenditures or it can also show that that a post maybe achieved zero impressions but still a very high expenditure and it can maybe be recorded as a significant loss for the advertising budget.

## Use Overall cost per impression to impute missing impression values

In [31]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation_filtered = (data_for_imputation.loc[(data_for_imputation['fbig_imp']>0)&
                                                            (data_for_imputation['fbig_spend']>0)])


cost_per_imp_for_imputation = (data_for_imputation_filtered['fbig_spend'].sum()/
             data_for_imputation_filtered['fbig_imp'].sum()
            )

print('Cost per Impression : ' + str(cost_per_imp_for_imputation))

Cost per Impression : 0.006257490152210633


### The dataset is filtered by this code to only contain rows with Facebook Big Spend and impressions greater than 0.  The average cost per impression is then determined by dividing the total spend in this filtered data by the total number of impressions.  It publishes and shows the estimated cost per impression at the end.

In [32]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation['fbig_imp'] = (np.where((data_for_imputation['fbig_spend']>0)
                                                                          & (data_for_imputation['fbig_imp']==0),
                                                                           data_for_imputation['fbig_spend']/cost_per_imp_for_imputation,
                                                                           data_for_imputation['fbig_imp']
                                                                 )
                                                                 )

data_for_imputation = data_for_imputation.set_index('week_starting_date')

### Only for rows where spend is greater than 0 and impressions are 0, this code estimates missing Facebook Big impressions using the cost per impression.  In order to facilitate time-series analysis, it then sets 'week_starting_date' as the DataFrame's index.

In [37]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation.loc[(data_for_imputation['fbig_spend']>0)&
                                               (data_for_imputation['fbig_imp']==0)]

,fbig_spend,fbig_imp
week_starting_date,,


### This code looks for weeks where no impressions were recorded despite Facebook ad spend.  Consistent data for such criteria is shown by an empty result, which indicates that there are no such examples to support this. 

In [34]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
print('Total Spend : ' + str(data_for_imputation['fbig_spend'].sum()))

Total Spend : 3271267.472244


### This code prints the total Facebook Big campaign spend by summing all the values in the fbig_spend column.










In [35]:
# STUDENT INPUT REQUIRED - Create similar code as above to print and check impressions total sum
print('Total Impressions : ' + str(data_for_imputation['fbig_imp'].sum()))

Total Impressions : 522776287.7242937


## Export processed data

In [36]:
# STUDENT COMMENT REQUIRED - write a short summary of what the python code in this cell is intended to do
data_for_imputation.to_csv(file_name)

### This code saves the cleaned and updated dataset to a CSV file at the specified export path for future use or analysis.